# Targeted training: two strategies

Does giving hard training sentences more weight beat giving the same weight to random sentences?

- **Strategy 1, margin percentile:** hard = lowest p% of the margin between the two highest probabilities. A copy of the trained baseline is trained a bit more, hard sentences weight 1.5.
- **Strategy 2, confidence band + clean labels:** hard = top probability between 0.4 and 0.6, and only sentences where all annotators agreed. Retrained from scratch, hard sentences weight K.

Both use the out-of-fold probabilities of the frozen FinBERT CNN from notebook 02 (`results/hard_samples/band_*.csv`).
Set `QUICK = True` to check that everything runs (1 seed, very few epochs); set it to `False` for the real runs.

## Settings

In [ ]:
QUICK = True    # True: only check that the code runs. False: the real experiment

SEEDS = [42, 43, 44, 45, 46]
PERCENTILES = [10, 20, 30, 40, 50]   # strategy 1
WEIGHT_S1 = 1.5                      # strategy 1
KS = [3, 5, 10]                      # strategy 2
AMBIGUOUS_WEIGHT = 0.5               # strategy 2 variant
EPOCHS = 20
FINE_TUNE_EPOCHS = 10
RUN_FINETUNED = False                # RQ2: repeat strategy 2 with fine-tuned embeddings (slow)

if QUICK:
    SEEDS = [42]
    PERCENTILES = [20]
    KS = [5]
    EPOCHS = 2
    FINE_TUNE_EPOCHS = 1

RESULTS_DIR = "../results/targeted"

## Imports

In [ ]:
import os
os.environ["HF_HOME"] = os.path.abspath("../.cache/huggingface")
os.environ["HF_HUB_CACHE"] = os.path.abspath("../.cache/huggingface/hub")

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from tensorflow import keras
from tensorflow.keras import layers
from transformers import BertModel, BertTokenizerFast

LABELS = ["negative", "neutral", "positive"]
MAX_LEN = 64
os.makedirs(RESULTS_DIR + "/probs", exist_ok=True)

## Data

In [ ]:
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
y_train = train_df.label.map({l: i for i, l in enumerate(LABELS)}).values
y_val = val_df.label.map({l: i for i, l in enumerate(LABELS)}).values

# FinBERT tokenizer + its embedding matrix (the CNN uses these as its input embeddings)
tokenizer = BertTokenizerFast.from_pretrained("yiyanghkust/finbert-pretrain")
finbert = BertModel.from_pretrained("yiyanghkust/finbert-pretrain")
embedding_matrix = finbert.embeddings.word_embeddings.weight.detach().numpy()
del finbert

def tokenize(df):
    return tokenizer(list(df.sentence), padding="max_length", truncation=True,
                     max_length=MAX_LEN, return_tensors="np")["input_ids"]

X_train = tokenize(train_df)
X_val = tokenize(val_df)

# out-of-fold probabilities (train) and baseline probabilities (val) from notebook 02
band_train = pd.read_csv("../results/hard_samples/band_train.csv")
band_val = pd.read_csv("../results/hard_samples/band_val.csv")
prob_cols = ["p_negative", "p_neutral", "p_positive"]
print(X_train.shape, X_val.shape, embedding_matrix.shape)

## All-agree flag

`Sentences_AllAgree.txt` from Financial PhraseBank: sentences where every annotator gave the same label. Matched by exact text.

In [ ]:
with open("../data/Sentences_AllAgree.txt", encoding="latin-1") as f:
    all_agree_sentences = set()
    for line in f:
        if line.strip():
            all_agree_sentences.add(line.rsplit("@", 1)[0].strip())

agree_train = train_df.sentence.str.strip().isin(all_agree_sentences).values
agree_val = val_df.sentence.str.strip().isin(all_agree_sentences).values
print("all-agree: train", agree_train.sum(), "/", len(train_df), "| val", agree_val.sum(), "/", len(val_df))

## Model

In [ ]:
def build_cnn(trainable=False):
    inputs = keras.Input(shape=(MAX_LEN,), dtype="int32")
    # start with zeros and copy FinBERT in afterwards; a Constant initializer makes clone_model run out of memory
    embedding = layers.Embedding(embedding_matrix.shape[0], embedding_matrix.shape[1],
                                 embeddings_initializer="zeros", trainable=trainable)
    x = embedding(inputs)
    x = layers.SpatialDropout1D(0.3)(x)
    x = layers.Conv1D(64, 3, activation="relu")(x)
    x = layers.GlobalMaxPooling1D()(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(3, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    embedding.set_weights([embedding_matrix])
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


class MacroF1(keras.callbacks.Callback):
    # macro F1 on val after every epoch, used for early stopping
    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(X_val, batch_size=32, verbose=0).argmax(axis=1)
        logs["val_macro_f1"] = f1_score(y_val, pred, average="macro")


def fit(model, weights, epochs, patience):
    early_stop = keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=patience,
                                               restore_best_weights=True)
    model.fit(X_train, keras.utils.to_categorical(y_train, 3), sample_weight=weights,
              validation_data=(X_val, keras.utils.to_categorical(y_val, 3)),
              epochs=epochs, batch_size=32, verbose=0, callbacks=[MacroF1(), early_stop])
    return model


def score(probs, hard_val):
    # accuracy and macro F1 on all val sentences and on the subsets
    pred = probs.argmax(axis=1)
    subsets = {"overall": np.ones(len(y_val), dtype=bool), "hard": hard_val, "nonhard": ~hard_val,
               "allagree": agree_val, "not_allagree": ~agree_val}
    scores = {}
    for name in subsets:
        mask = subsets[name]
        scores["accuracy_" + name] = accuracy_score(y_val[mask], pred[mask])
        scores["f1_" + name] = f1_score(y_val[mask], pred[mask], average="macro", zero_division=0)
    return scores


def save_rows(rows, name):
    # one csv per strategy; running a setting again replaces its old rows
    path = f"{RESULTS_DIR}/{name}.csv"
    new = pd.DataFrame(rows)
    if os.path.exists(path):
        new = pd.concat([pd.read_csv(path), new], ignore_index=True)
    key = [c for c in ["arm", "param", "embeddings", "seed"] if c in new.columns]
    new = new.drop_duplicates(subset=key, keep="last")
    new.to_csv(path, index=False)
    return new

## Strategy 1: margin percentile

Margin = highest probability - second highest. Hard = lowest p% of the train margins; the same cutoff is used on val.
The baseline is trained once, then for every seed a copy is trained further (lr 1e-4, up to 10 epochs) with the hard sentences (targeted) or the same number of random sentences (random control) at weight 1.5.

In [ ]:
def margins(probs):
    sorted_probs = np.sort(probs, axis=1)
    return sorted_probs[:, -1] - sorted_probs[:, -2]

margin_train = margins(band_train[prob_cols].values)
margin_val = margins(band_val[prob_cols].values)

keras.utils.set_random_seed(42)
baseline = fit(build_cnn(), None, EPOCHS, patience=5)
print("baseline val macro F1:", round(score(baseline.predict(X_val, verbose=0), band_val.is_hard.values)["f1_overall"], 3))

In [ ]:
def fine_tune(weighted_idx, seed):
    keras.utils.set_random_seed(seed)
    model = keras.models.clone_model(baseline)
    model.set_weights(baseline.get_weights())
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
    weights = np.ones(len(y_train))
    weights[weighted_idx] = WEIGHT_S1
    return fit(model, weights, FINE_TUNE_EPOCHS, patience=1)


rows = []
for p in PERCENTILES:
    cutoff = np.percentile(margin_train, p)
    hard_idx = np.where(margin_train <= cutoff)[0]
    hard_val = margin_val <= cutoff
    print(f"P{p}: {len(hard_idx)} hard train, {hard_val.sum()} hard val (cutoff {cutoff:.3f})")

    for seed in SEEDS:
        random_idx = np.random.default_rng(seed).choice(len(y_train), len(hard_idx), replace=False)
        for arm, idx in [("targeted", hard_idx), ("random", random_idx)]:
            probs = fine_tune(idx, seed).predict(X_val, batch_size=32, verbose=0)
            np.save(f"{RESULTS_DIR}/probs/s1_{arm}_p{p}_s{seed}.npy", probs)
            row = {"arm": arm, "param": p, "seed": seed, "n_weighted": len(idx), "n_hard_val": int(hard_val.sum())}
            row.update(score(probs, hard_val))
            rows.append(row)
            print(f"   {arm} seed={seed}: f1={row['f1_overall']:.3f} hard={row['f1_hard']:.3f}")

strategy1 = save_rows(rows, "strategy1")

## Strategy 2: confidence band + clean labels

Hard = top probability between 0.4 and 0.6 (the `is_hard` column of the band files). Only the hard sentences where all annotators agreed are up-weighted (clean-hard). Every run is trained from scratch.

| arm | sentences with weight K |
|---|---|
| no_weight | none (reference) |
| clean_hard | hard and all-agree |
| random | same number, random from all train |
| clean_random | same number, random from all-agree sentences that are not hard |
| variant | clean_hard gets K, hard sentences without full agreement get 0.5 |

In [ ]:
hard_train = band_train.is_hard.values
hard_val_s2 = band_val.is_hard.values

clean_hard = np.where(hard_train & agree_train)[0]
ambiguous_hard = np.where(hard_train & ~agree_train)[0]
clean_easy = np.where(~hard_train & agree_train)[0]
print("hard train:", hard_train.sum(), "| clean-hard:", len(clean_hard), "| ambiguous hard:", len(ambiguous_hard),
      "| hard val:", hard_val_s2.sum())


def strategy2_weights(arm, k, seed):
    rng = np.random.default_rng(seed)
    weights = np.ones(len(y_train))
    if arm == "clean_hard":
        weights[clean_hard] = k
    elif arm == "random":
        weights[rng.choice(len(y_train), len(clean_hard), replace=False)] = k
    elif arm == "clean_random":
        weights[rng.choice(clean_easy, len(clean_hard), replace=False)] = k
    elif arm == "variant":
        weights[ambiguous_hard] = AMBIGUOUS_WEIGHT
        weights[clean_hard] = k
    return weights


def run_strategy2(arm, k, seed, trainable=False):
    keras.utils.set_random_seed(seed)
    model = fit(build_cnn(trainable), strategy2_weights(arm, k, seed), EPOCHS, patience=5)
    probs = model.predict(X_val, batch_size=32, verbose=0)
    embeddings = "finetuned" if trainable else "frozen"
    np.save(f"{RESULTS_DIR}/probs/s2_{arm}_k{k}_{embeddings}_s{seed}.npy", probs)
    row = {"arm": arm, "param": k, "embeddings": embeddings, "seed": seed}
    row.update(score(probs, hard_val_s2))
    print(f"   {arm} k={k} {embeddings} seed={seed}: f1={row['f1_overall']:.3f} hard={row['f1_hard']:.3f}")
    return row

In [ ]:
rows = []
for seed in SEEDS:
    rows.append(run_strategy2("no_weight", 1, seed))
for k in KS:
    for arm in ["clean_hard", "random", "clean_random", "variant"]:
        for seed in SEEDS:
            rows.append(run_strategy2(arm, k, seed))
strategy2 = save_rows(rows, "strategy2")

### RQ2: fine-tuned embeddings (optional, slow)

Same as above for no_weight, clean_hard and random at K = 5, with the FinBERT embeddings trained too. Set `RUN_FINETUNED = True` in the settings.

In [ ]:
if RUN_FINETUNED:
    rows = []
    for arm, k in [("no_weight", 1), ("clean_hard", 5), ("random", 5)]:
        for seed in SEEDS:
            rows.append(run_strategy2(arm, k, seed, trainable=True))
    strategy2 = save_rows(rows, "strategy2")
else:
    print("skipped (RUN_FINETUNED = False)")

## Compare

Mean and std over the seeds, and targeted minus its control seed by seed (both use the same seeds). A difference smaller than the std is not an effect.

In [ ]:
cols = ["f1_overall", "f1_hard", "f1_nonhard", "f1_allagree", "f1_not_allagree"]

s1 = pd.read_csv(f"{RESULTS_DIR}/strategy1.csv")
s1.groupby(["param", "arm"])[cols].agg(["mean", "std"]).round(3)

In [ ]:
s2 = pd.read_csv(f"{RESULTS_DIR}/strategy2.csv")
s2.groupby(["embeddings", "arm", "param"])[cols].agg(["mean", "std"]).round(3)

In [ ]:
def paired_diff(df, target, control, param):
    a = df[(df.arm == target) & (df.param == param)].sort_values("seed").f1_overall.values
    b = df[(df.arm == control) & (df.param == param)].sort_values("seed").f1_overall.values
    if len(a) == 0 or len(a) != len(b):
        return
    diff = a - b
    std = diff.std(ddof=1) if len(diff) > 1 else float("nan")
    print(f"{target} - {control}, param={param}: mean {diff.mean():+.3f}, std {std:.3f}, per seed {np.round(diff, 3).tolist()}")

print("Strategy 1")
for p in sorted(s1.param.unique()):
    paired_diff(s1, "targeted", "random", p)

print("\nStrategy 2 (frozen)")
frozen = s2[s2.embeddings == "frozen"]
for k in sorted(frozen[frozen.arm != "no_weight"].param.unique()):
    for control in ["random", "clean_random"]:
        paired_diff(frozen, "clean_hard", control, k)